# Laden der Daten


In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import fashion_mnist

# Datensatz laden
(x_train, y_train), (x_test, y_test_original) = fashion_mnist.load_data()

# Pixelwerte auf den Bereich [0, 1] skalieren
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Convolution Layer arbeiten besser, auf echten Bildern mit Farbkanal
# --> Wir fügen den Farbkanal hinzu
print("Ohne Kanal:", x_train.shape)
x_train = x_train[..., None]
x_test = x_test[..., None]
print("Mit Kanal:", x_train.shape)

# One Hot encoding
y_train = to_categorical(y_train)
y_test = to_categorical(y_test_original)

# Neuronales Netz erzeugen

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, MaxPooling2D, Dropout

model = Sequential()
model.add(Input(shape=(28, 28, 1)))

model.add(Conv2D(32, (3, 3), padding="same", activation="relu"))
model.add(Conv2D(32, (3, 3), padding="same", activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))

model.add(Conv2D(64, (3, 3), padding="same", activation="relu"))
model.add(Conv2D(64, (3, 3), padding="same", activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))

model.add(Flatten())
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.4))
model.add(Dense(10, activation="softmax"))

model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

# Neuronales Netz trainieren

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor="val_loss",
                   patience=3,
                   restore_best_weights=True)

history = model.fit(x_train, y_train,
          epochs=50,
          batch_size=128,
          validation_split=0.1,
          callbacks=[es])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lossMonitor = np.array(history.history['loss'])
valLossMonitor = np.array(history.history['val_loss'])
counts = np.arange(lossMonitor.shape[0])

fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
ax.plot(counts, valLossMonitor, 'r:', label='Validierungsdaten')
ax.set_xlabel('Lernzyklus')
ax.set_ylabel('Fehler')
ax.set_xticks(counts[::1])
#ax.set_xticks(counts)
ax.set_ylim(0, 0.50)
ax.legend()
plt.show()

In [ ]:
model.evaluate(x_test, y_test)

# Confusion Matrix erstellen

In [ ]:
from sklearn import metrics
import numpy as np

# Wir berechnen alle Vorhersagen mit einem Methodenaufruf
predictions = model.predict(x_test)

# vorhersagen ist ein 10000x10 Feld. Mit axis können wir
# wählen auf welcher Achse das argmax angewendet wird
predictions = np.argmax(predictions, axis = 1)

# Wir berechnen die Confusion Matrix
confusion_matrix = metrics.confusion_matrix(y_test_original, predictions)

In [ ]:
fashion_labels = [
    "T-Shirt/Top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle Boot"
]

cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix,
    display_labels=fashion_labels
)

fig, ax = plt.subplots(figsize=(10, 10))
cm_display.plot(ax=ax, xticks_rotation=45)
plt.show()